In [ ]:
import datetime
import glob
from pprint import pprint

import pandas as pd
import pyarrow.parquet as pq
import numpy as np

# Path to Argo BGC 'ALL'
parquet_dir = "../../../shared/go-bgc-2026/data/CrocoLake/BGC_ARGO/"
# Setting up parquet schema
BGC_schema = pq.read_schema(parquet_dir+"_common_metadata")

dataset = pq.ParquetDataset(
parquet_dir, 
schema=BGC_schema
)
schema = dataset.schema

heatwave_list = pd.read_csv("../../MHW_list.csv")
heatwave_list

In [ ]:
heatwave_dict = {
    "name":[],
    "data":[]
}
for i, heatwave_name in enumerate(heatwave_list.name):
    # Boundaries
    lat0 = heatwave_list.lat0[i]
    lat1 = heatwave_list.lat1[i]
    lon0 = heatwave_list.lon0[i]
    lon1 = heatwave_list.lon1[i]
    try:
            
        start_time = datetime.datetime.strptime(str(heatwave_list.date_start[i]), "%Y-%m-%d")
        end_time = datetime.datetime.strptime(str(heatwave_list.date_end[i]), "%Y-%m-%d")
        
        filter_coords_time = [
            ("JULD",">",start_time), ("JULD","<",end_time),
            ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
            ("LONGITUDE",">",lon0), ("LONGITUDE","<",lon1)
        ]
    
        ds = pq.ParquetDataset(parquet_dir, schema=BGC_schema, filters=filter_coords_time)
        df = ds.read().to_pandas()
        
        heatwave_dict["name"].append(heatwave_name)
        heatwave_dict["data"].append(df)
    except ValueError:
        print(f"Value Error for heatwave {heatwave_name}")
        heatwave_dict["name"].append(heatwave_name)
        heatwave_dict["data"].append(np.nan)   

In [ ]:
heatwave_dict_df = pd.DataFrame(heatwave_dict)

In [ ]:
heatwave_dict_df